# Packing guardrail failure demo

This notebook answers one question: **when a packing rule is broken, does the guardrail reject the plan?**

We use small simulated orders so every failure is obvious from the inputs. Each scenario shows the items, the carton, why the plan must fail, and the actual validator result. A scenario passes only when the expected guardrail blocks the invalid plan.


In [ ]:
from copy import deepcopy
from dataclasses import replace
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from bin_packing_3d.models import Box, Item, Orientation, PackedBox, PackingPlan, PhysicalItem, Placement, Position
from bin_packing_3d.rules import effective_max_fill_pct, normalize_config, usable_dimensions
from bin_packing_3d.validate import validate_plan

def make_case(item_dimensions=(10,10,7), count=2, item_weight=5, vertical_rotation=False,
              box_dimensions=(20,20,20), max_weight=10, positions=None, config=None):
    cfg = normalize_config(config or {
        "bin_buffer": {"length": 0, "width": 0, "height": 6},
        "max_fill_pct": 100, "high_item_count_threshold": 6,
        "high_item_count_max_fill_pct": 70,
    })
    source = Item("A", *item_dimensions, item_weight, count, vertical_rotation)
    items = [PhysicalItem(f"A#{i+1}", source) for i in range(count)]
    box = Box("DemoBox", *box_dimensions, max_weight)
    positions = positions or [Position(i*10,0,0) for i in range(count)]
    placements = [Placement(item.instance_id,item.code,"DemoBox-1",Orientation(*item_dimensions),pos)
                  for item,pos in zip(items,positions)]
    plan = PackingPlan([PackedBox("DemoBox","DemoBox-1",usable_dimensions(box,cfg),placements)])
    return plan, items, [box], cfg

def expect_guardrail(name, expected_code, plan, items, boxes, cfg):
    result = validate_plan(plan, items, boxes, cfg)
    codes = {e.code for e in result.errors}
    assert not result.valid and expected_code in codes, f"{name}: expected {expected_code}, got {codes}"
    print(f"GUARDRAIL PASS — invalid plan rejected: {expected_code}")
    for error in result.errors:
        print(f"  {error.code}: {error.message}")
    return {"scenario": name, "expected_guardrail": expected_code, "result": "PASS"}

guardrail_results = []


## Scenario 1 — Carton weight exceeded

**Simulated order:** 2 × item A, each **10 × 10 × 7 mm** and **5.01 kg**. Total order weight = **10.02 kg**.

**Available carton:** DemoBox = **20 × 20 × 20 mm**, maximum **10 kg**.

**Why this must fail:** the items geometrically fit, but their combined weight is above the carton limit.


In [ ]:
plan, items, boxes, cfg = make_case(item_weight=5.01)
guardrail_results.append(expect_guardrail("Weight limit", "weight", plan, items, boxes, cfg))


## Scenario 2 — Item exceeds usable carton boundary

**Simulated order:** 1 × item A = **10 × 10 × 7 mm**.

**Available carton:** DemoBox = **20 × 20 × 20 mm**, but the configured **6 mm height buffer** leaves only **14 mm usable height**.

**Simulated placement:** item starts at Z = **7.1 mm**, so it ends at **14.1 mm**.

**Why this must fail:** the item exceeds the usable carton height by 0.1 mm.


In [ ]:
plan, items, boxes, cfg = make_case(count=1,item_weight=1,positions=[Position(0,0,7.1)])
guardrail_results.append(expect_guardrail("Usable carton boundary", "boundary", plan, items, boxes, cfg))


## Scenario 3 — Rotation not allowed

**Simulated order:** 1 × item A = **10 × 10 × 7 mm**, with `VerticalRotation = 0`.

**Available carton:** DemoBox = **20 × 20 × 20 mm**.

**Simulated placement:** force the item orientation to **10 × 7 × 10 mm**.

**Why this must fail:** the 7 mm height has been rotated onto a side even though vertical rotation is disabled.


In [ ]:
plan, items, boxes, cfg = make_case(count=1,item_weight=1)
plan.packed_boxes[0].placements[0] = replace(plan.packed_boxes[0].placements[0], orientation=Orientation(10,7,10))
guardrail_results.append(expect_guardrail("Allowed rotation", "orientation", plan, items, boxes, cfg))


## Scenario 4 — Items overlap

**Simulated order:** 2 × item A, each **10 × 10 × 7 mm**.

**Available carton:** DemoBox = **20 × 20 × 20 mm**.

**Simulated placements:** first item starts at X = **0 mm**; second starts at X = **9.9 mm**.

**Why this must fail:** the first item occupies X = 0–10 mm, so the second item overlaps it by 0.1 mm.


In [ ]:
plan, items, boxes, cfg = make_case(positions=[Position(0,0,0),Position(9.9,0,0)])
guardrail_results.append(expect_guardrail("Non-overlap", "overlap", plan, items, boxes, cfg))


## Scenario 5 — More than 6 items exceed the 70% fill cap

**Simulated order:** 8 × item A, each **10 × 10 × 1 mm** = **100 mm³** each. Total item volume = **800 mm³**.

**Available carton:** DemoBox = **10 × 10 × 10 mm** = **1,000 mm³** usable volume.

**Configured rule:** orders with more than 6 physical items may use at most **70%** of carton volume.

**Why this must fail:** 800 / 1,000 = **80% fill**, above the 70% limit.


In [ ]:
fill_cfg={"bin_buffer":{"length":0,"width":0,"height":0},"max_fill_pct":100,
          "high_item_count_threshold":6,"high_item_count_max_fill_pct":70}
plan, items, boxes, cfg = make_case(item_dimensions=(10,10,1),count=8,item_weight=0.1,
    box_dimensions=(10,10,10),max_weight=10,positions=[Position(0,0,z) for z in range(8)],config=fill_cfg)
assert effective_max_fill_pct(len(items),cfg) == 70
guardrail_results.append(expect_guardrail("70% fill cap", "fill_cap", plan, items, boxes, cfg))


## Scenario 6 — Physical item missing from the packing plan

**Simulated order:** quantity = **2**, producing physical items **A#1** and **A#2**.

**Available carton:** DemoBox = **20 × 20 × 20 mm**.

**Simulated bad result:** only A#1 is returned in the carton; A#2 disappears from the plan.

**Why this must fail:** every physical item in the order must be accounted for exactly once.


In [ ]:
plan, items, boxes, cfg = make_case()
plan.packed_boxes[0].placements.pop()
guardrail_results.append(expect_guardrail("Physical item accounting", "missing_item", plan, items, boxes, cfg))


## Scenario 7 — Carton not in the configured catalogue

**Simulated order:** 1 × item A = **10 × 10 × 7 mm**.

**Configured carton catalogue:** contains only `DemoBox`.

**Simulated bad result:** the packing plan claims it used `NotInCatalogue`.

**Why this must fail:** the solver may only return configured carton types.


In [ ]:
plan, items, boxes, cfg = make_case(count=1,item_weight=1)
plan.packed_boxes[0].box_code = "NotInCatalogue"
guardrail_results.append(expect_guardrail("Carton catalogue", "unknown_box", plan, items, boxes, cfg))


## Scenario 8 — Packing plan bypasses the configured 6 mm buffer

**Simulated order:** 1 × item A = **10 × 10 × 7 mm**.

**External carton:** DemoBox = **20 × 20 × 20 mm**.

**Configured usable carton:** **20 × 20 × 14 mm** after the 6 mm height buffer.

**Simulated bad result:** the plan claims the full **20 × 20 × 20 mm** is usable.

**Why this must fail:** a packing strategy cannot bypass the configured carton buffer.


In [ ]:
plan, items, boxes, cfg = make_case(count=1,item_weight=1)
plan.packed_boxes[0].usable_dimensions = Orientation(20,20,20)
guardrail_results.append(expect_guardrail("Configured usable dimensions", "usable_dimensions", plan, items, boxes, cfg))


## Scenario 9 — Placement points to the wrong carton

**Simulated order:** 1 × item A.

**Packed carton instance:** `DemoBox-1`.

**Simulated bad result:** the item placement says it belongs to `WrongBox-1`.

**Why this must fail:** every placement must reference the carton that actually contains it.


In [ ]:
plan, items, boxes, cfg = make_case(count=1,item_weight=1)
plan.packed_boxes[0].placements[0] = replace(plan.packed_boxes[0].placements[0],box_instance_id="WrongBox-1")
guardrail_results.append(expect_guardrail("Carton reference", "box_reference", plan, items, boxes, cfg))


## Scenario 10 — Placement changes the item identity

**Simulated order:** physical item `A#1` has item code `A`.

**Simulated bad result:** its placement is returned with item code `WRONG`.

**Why this must fail:** the packing plan cannot change the identity of an ordered item.


In [ ]:
plan, items, boxes, cfg = make_case(count=1,item_weight=1)
plan.packed_boxes[0].placements[0] = replace(plan.packed_boxes[0].placements[0],item_code="WRONG")
guardrail_results.append(expect_guardrail("Item identity", "item_code", plan, items, boxes, cfg))


## Scenario 11 — Negative XYZ position

**Simulated order:** 1 × item A.

**Simulated placement:** X = **−0.1 mm**, Y = 0, Z = 0.

**Why this must fail:** carton coordinates start at zero; a negative coordinate places part of the item outside the carton.


In [ ]:
plan, items, boxes, cfg = make_case(count=1,item_weight=1)
plan.packed_boxes[0].placements[0] = replace(plan.packed_boxes[0].placements[0],position=Position(-0.1,0,0))
guardrail_results.append(expect_guardrail("Non-negative XYZ", "negative_coordinate", plan, items, boxes, cfg))


## Guardrail result

The demo succeeds only if **every deliberately invalid scenario is rejected by the expected guardrail**.


In [ ]:
assert all(row["result"] == "PASS" for row in guardrail_results)
assert len(guardrail_results) == 11
print(f"ALL GUARDRAILS PASS — {len(guardrail_results)}/{len(guardrail_results)} invalid scenarios rejected as expected")
